<a href="https://colab.research.google.com/github/AntonTian/Multimodal-PreThesis/blob/main/NLP_Fine_Tuning_%2B_Visual_Representation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
import torch
print("Is the GPU awake and connected to Python?:", torch.cuda.is_available())

In [ ]:
# 1. Install necessary libraries
!pip install transformers datasets evaluate accelerate scikit-learn

import numpy as np
import shutil
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
from google.colab import drive

print("="*50)
print("Stage 1: Data Prep (The Reddit Sarcasm Dataset)")
print("="*50)

# 2. Download the SARC dataset directly from Hugging Face
print("Downloading conversational Reddit dataset from Hugging Face...")
raw_dataset = load_dataset("marcbishara/sarcasm-on-reddit")

# 3. Format the data for our pipeline
# The dataset calls the sentence column 'comment', but our tokenizer looks for 'text'
raw_dataset = raw_dataset.rename_column("comment", "text")

# 4. Sampling & Splitting
sampled_dataset = raw_dataset['sft_train'].shuffle(seed=42).select(range(20000))

# Split into 80% training and 20% testing
hf_dataset = sampled_dataset.train_test_split(test_size=0.2)

print(f"\nDataset loaded successfully! Total samples: {len(sampled_dataset)}")

print("\n" + "="*50)
print("Stage 2: Tokenization & Model Loading")
print("="*50)

# 5. Tokenization
print("Downloading RoBERTa Base Tokenizer...")
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

# 6. Load the Base Model
print("Loading RoBERTa Base Model with binary classification head...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 7. Define Evaluation Metrics
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

print("\n" + "="*50)
print("Stage 3: Fine-Tuning")
print("="*50)

# 8. Training Arguments
training_args = TrainingArguments(
    output_dir="./sarcasm_results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

# 9. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

# 10. Start Fine-Tuning
print("Training initiated. Utilizing Cloud GPU...")
trainer.train()

print("\n" + "="*50)
print("Stage 4: Saving & Backup")
print("="*50)

# 11. Save your custom model locally in the Colab environment
local_dir = "./my_sarcasm_model"
model.save_pretrained(local_dir)
tokenizer.save_pretrained(local_dir)
print(f"Model saved temporarily to '{local_dir}'.")

# 12. Backup permanently to Google Drive
print("\nConnecting to Google Drive for permanent backup...")
drive.mount('/content/drive')

drive_dir = '/content/drive/MyDrive/my_sarcasm_model'
print(f"Copying model to {drive_dir}...")

# Remove the directory if it already exists from a previous run
shutil.rmtree(drive_dir, ignore_errors=True)
shutil.copytree(local_dir, drive_dir)

print("\nSuccess! Your conversational Sarcasm Gatekeeper is permanently saved in your Google Drive.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**F1-Score Representation + Accuracy**

In [ ]:
!pip install datasets transformers scikit-learn matplotlib seaborn

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from datasets import load_dataset
from sklearn.metrics import confusion_matrix
from transformers import pipeline
import os

# Force legacy Keras to avoid version conflicts
os.environ["TF_USE_LEGACY_KERAS"] = "1"

sns.set_theme(style="whitegrid", font_scale=1.1)

print("="*50)
print("[1/3]: Recreating the 4,000-Sample Test Set & Running Inference on Custom Mode")
print("="*50)

print("Downloading Reddit dataset from Hugging Face...")
raw_dataset = load_dataset("marcbishara/sarcasm-on-reddit")
raw_dataset = raw_dataset.rename_column("comment", "text")

sampled_dataset = raw_dataset['sft_train'].shuffle(seed=42).select(range(20000))
hf_dataset = sampled_dataset.train_test_split(test_size=0.2, seed=42)

real_test_texts = [str(text) if text is not None else "" for text in hf_dataset['test']['text']]
true_labels = hf_dataset['test']['label']

print(f"Successfully loaded {len(real_test_texts)} real test sentences!")

print("Loading Custom Sarcasm Gatekeeper from Google Drive...")
sarcasm_model = pipeline("text-classification", model="/content/drive/MyDrive/my_sarcasm_model", device=0)

print("Scanning test set (this will take a few seconds)...")
predictions = sarcasm_model(real_test_texts, batch_size=32, truncation=True, max_length=128)

predicted_labels = [1 if p['label'] == 'LABEL_1' else 0 for p in predictions]

print("\n" + "="*50)
print("n[1/3]: Generating Authentic Figure 1")
print("="*50)

cm_data = confusion_matrix(true_labels, predicted_labels)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_data, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=['Literal (Pred)', 'Sarcasm (Pred)'],
            yticklabels=['Literal (Actual)', 'Sarcasm (Actual)'])

plt.title('Fig 1. Sarcasm Gatekeeper Confusion Matrix\n(Actual Model Inference on 4,000 Samples)', pad=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_real_confusion_matrix.png', dpi=300)
plt.show()

print("\n[2/3]: Extracting Real Training Loss from Keras Engine...")
np.random.seed(42)
tf.random.set_seed(42)
inputs, targets = [], []

for _ in range(5000):
    text_v, face_a = np.random.uniform(0.0, 1.0), np.random.uniform(0.0, 1.0)
    if text_v < 0.4 and face_a > 0.6:
        target_v, target_e = text_v, text_v + 0.1
    elif text_v > 0.6 and face_a < 0.4:
        target_v, target_e = text_v, text_v - 0.1
    else:
        target_v, target_e = text_v, face_a
    inputs.append([text_v, face_a])
    targets.append([np.clip(target_v, 0.0, 1.0), np.clip(target_e, 0.0, 1.0)])

fusion_model = Sequential([
    Input(shape=(2,)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(2, activation='linear')
])
fusion_model.compile(optimizer='adam', loss='mse')

real_history = fusion_model.fit(np.array(inputs), np.array(targets), epochs=15, batch_size=64, verbose=0)
real_loss = real_history.history['loss']
epochs = np.arange(1, len(real_loss) + 1)

plt.figure(figsize=(8, 4))
plt.plot(epochs, real_loss, marker='o', linestyle='-', color='indigo', linewidth=2, label='Real MSE Loss')
plt.title('Fig 2. Neural Fusion Engine Training Convergence', pad=15, fontweight='bold')
plt.xlabel('Training Epochs')
plt.ylabel('Loss (Mean Squared Error)')
plt.xticks(epochs)
plt.legend()
plt.tight_layout()
plt.savefig('fig2_real_training_curve.png', dpi=300)
plt.show()

print("\n[3/3] Mapping Real Spotify Features...")
try:
    df_spotify = pd.read_csv('dataset.csv')

    val_col = 'valence' if 'valence' in df_spotify.columns else 'Valence'
    eng_col = 'energy' if 'energy' in df_spotify.columns else 'Energy'

    plt.figure(figsize=(7, 6))
    sns.scatterplot(x=val_col, y=eng_col, data=df_spotify, alpha=0.5, color='teal', edgecolor=None, s=15)

    plt.axvline(0.5, color='gray', linestyle='--', linewidth=1)
    plt.axhline(0.5, color='gray', linestyle='--', linewidth=1)

    plt.text(0.15, 0.85, 'Angry / Anxious', fontsize=12, alpha=0.9, fontweight='bold', bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))
    plt.text(0.70, 0.85, 'Happy / Excited', fontsize=12, alpha=0.9, fontweight='bold', bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))
    plt.text(0.15, 0.15, 'Sad / Exhausted', fontsize=12, alpha=0.9, fontweight='bold', bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))
    plt.text(0.70, 0.15, 'Calm / Relaxed', fontsize=12, alpha=0.9, fontweight='bold', bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))

    plt.title('Fig 3. Actual Spotify Database on Russell\'s Circumplex', pad=15, fontweight='bold')
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig('fig3_real_circumplex_map.png', dpi=300)
    plt.show()

except FileNotFoundError:
    print("Error: 'dataset.csv' not found. Please upload it to your Colab environment to generate Fig 3.")

print("\nSuccess! Authentic, data-driven figures saved.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
from transformers import pipeline
from datasets import load_dataset
import os

# Force legacy Keras
os.environ["TF_USE_LEGACY_KERAS"] = "1"
sns.set_theme(style="whitegrid", font_scale=1.1)

print("="*50)
print("[1/4]: Loading Data & Models...")
print("="*50)

# Load data
raw_dataset = load_dataset("marcbishara/sarcasm-on-reddit")
raw_dataset = raw_dataset.rename_column("comment", "text")
hf_dataset = raw_dataset['sft_train'].shuffle(seed=42).select(range(20000)).train_test_split(test_size=0.2, seed=42)

real_test_texts = [str(text) if text is not None else "" for text in hf_dataset['test']['text']]
y_true = hf_dataset['test']['label']

# Load models
sarcasm_model = pipeline("text-classification", model="/content/drive/MyDrive/my_sarcasm_model", device=0)
goemotions_model = pipeline("text-classification", model="SamLowe/roberta-base-go_emotions", top_k=1, device=0)

print("\n" + "="*50)
print("[2/4]: Running Batched Inferences (4,000 samples)")
print("="*50)

base_raw = goemotions_model(real_test_texts, batch_size=32, truncation=True, max_length=128)
sarc_raw = sarcasm_model(real_test_texts, batch_size=32, truncation=True, max_length=128)

base_predictions = []
gatekeeper_predictions = []
positive_emotions = ['joy', 'amusement', 'excitement', 'love', 'admiration', 'optimism', 'relief', 'pride', 'gratitude']

for i in range(len(real_test_texts)):
    # Baseline Logic
    is_base_positive = base_raw[i][0]['label'] in positive_emotions
    base_predictions.append(0 if is_base_positive else 1)

    # Gatekeeper Logic
    is_sarcastic = sarc_raw[i]['label'] == 'LABEL_1'
    if is_sarcastic:
        gatekeeper_predictions.append(1)
    else:
        gatekeeper_predictions.append(0 if is_base_positive else 1)

print("\n" + "="*50)
print("[3/4]: Generating Classification Reports")
print("="*50)

base_report = classification_report(y_true, base_predictions, output_dict=True)
gk_report = classification_report(y_true, gatekeeper_predictions, output_dict=True)

print("\n--- Baseline GoEmotions ---")
print(classification_report(y_true, base_predictions))
print("\n--- Two-Stage Pipeline (Optimized) ---")
print(classification_report(y_true, gatekeeper_predictions))

print("\n" + "="*50)
print("[4/4]: Plotting the Comprehensive 4-Metric Chart")
print("="*50)

labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

base_scores = [
    base_report['accuracy'] * 100,
    base_report['macro avg']['precision'] * 100,
    base_report['macro avg']['recall'] * 100,
    base_report['macro avg']['f1-score'] * 100
]

gk_scores = [
    gk_report['accuracy'] * 100,
    gk_report['macro avg']['precision'] * 100,
    gk_report['macro avg']['recall'] * 100,
    gk_report['macro avg']['f1-score'] * 100
]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, base_scores, width, label='Base GoEmotions', color='lightcoral')
rects2 = ax.bar(x + width/2, gk_scores, width, label='Two-Stage Pipeline (Yours)', color='teal')

ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Fig 4. Comprehensive Model Evaluation on Modality Incongruity', pad=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontweight='bold')
ax.set_ylim(0, 110)

ax.legend(loc='upper right', bbox_to_anchor=(1, 1))

ax.bar_label(rects1, fmt='%.1f%%', padding=3)
ax.bar_label(rects2, fmt='%.1f%%', padding=3)

plt.tight_layout()
plt.savefig('fig4_comprehensive_metrics.png', dpi=300)
plt.show()